# Punto 5 (Libre): Diagnóstico Médico con Redes Neuronales

**Enunciado propuesto:** Utilizando TensorFlow/Keras y datos médicos reales, construya una Red Neuronal capaz de predecir si un tumor de cáncer de mama es 'Maligno' o 'Benigno' basándose en características celulares. Entrene el modelo, analice su precisión y realice una predicción sobre un caso clínico de prueba.

In [ ]:
# 1. IMPORTACIÓN DE LIBRERÍAS
import tensorflow as tf
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("Librerías importadas correctamente.")

: 

## Paso 1: Carga de Datos Clínicos
Usaremos el **Breast Cancer Wisconsin Dataset**, que contiene 569 casos médicos reales con 30 mediciones del núcleo de las células (radio, textura, área, etc.). Es ultra ligero e instantáneo de cargar.

In [ ]:
print("Cargando el dataset médico...")
datos = load_breast_cancer()
X = datos.data
y = datos.target # 0 = Maligno, 1 = Benigno
nombres_caracteristicas = datos.feature_names
nombres_clases = datos.target_names

print(f"Total de pacientes: {X.shape[0]}")
print(f"Características evaluadas por paciente: {X.shape[1]}")

: 

## Paso 2: Preprocesamiento
Separamos los datos en entrenamiento y prueba, y aplicamos estandarización, ya que las mediciones celulares tienen escalas muy distintas.

In [ ]:
# División 80% entrenamiento, 20% prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Escalamiento
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Datos divididos y estandarizados con éxito.")

: 

## Paso 3: Construcción de la Red Neuronal
Usamos **TensorFlow/Keras** para crear una red de dos capas ocultas. Al ser un problema de clasificación binaria (Maligno vs Benigno), la capa final tendrá 1 sola neurona con activación Sigmoid.

In [ ]:
modelo = tf.keras.Sequential([
    tf.keras.layers.Dense(16, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid') # Salida entre 0 y 1
])

modelo.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

modelo.summary()

: 

## Paso 4: Entrenamiento ultrarrápido
Entrenamos el modelo.

In [ ]:
print("Iniciando el entrenamiento...")
# verbose=0 hace que el entrenamiento sea silencioso en lugar de imprimir línea por línea
historial = modelo.fit(X_train_scaled, y_train, epochs=30, validation_split=0.2, verbose=0)
print("¡Entrenamiento Finalizado en milisegundos!")

# Evaluación general
perdida, precision = modelo.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\n-> Precisión del modelo en diagnósticos médicos nuevos: {precision*100:.2f}%")

: 

## Paso 5: Predicción de un Caso Clínico Real
Simulamos la llegada de un paciente al hospital.

In [ ]:
# Tomamos el caso de un paciente del conjunto de prueba
paciente_idx = 0
caso_medico = X_test[paciente_idx]
diagnostico_real = nombres_clases[y_test[paciente_idx]]

print("\n" + "="*50)
print(" MEDIDAS CELULARES DEL PACIENTE (Resumen): ")
for i in range(4):
    print(f"  - {nombres_caracteristicas[i]}: {caso_medico[i]:.2f}")
print("  - ... (y otras 26 mediciones)")
print("="*50)

# Importante escalar el dato antes de pasarlo a TensorFlow
caso_medico_escalado = scaler.transform([caso_medico])

# Predicción
probabilidad = modelo.predict(caso_medico_escalado, verbose=0)[0][0]
# La documentación indica que > 0.5 es Benigno (1) y <= 0.5 es Maligno (0)
diagnostico_ia = nombres_clases[1] if probabilidad > 0.5 else nombres_clases[0]

print(f"\n=> DIAGNÓSTICO DE LA INTELIGENCIA ARTIFICIAL: Tumor {diagnostico_ia}")
print(f"=> DIAGNÓSTICO REAL (Laboratorio): Tumor {diagnostico_real}")

if diagnostico_ia == diagnostico_real:
    print("\n=> ¡La IA acertó el diagnóstico!")
else:
    print("\n=> La IA falló el diagnóstico.")

: 